In [1]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [54]:
class_reports = glob('./logs/hires_simclr_tests_finetune2/*/*/*/*/classification_report.csv')
class_reports

['./logs/hires_simclr_tests_finetune2/hires_simclr/decoder_only/frozen_encoder/dae/classification_report.csv',
 './logs/hires_simclr_tests_finetune2/hires_simclr/full_model/frozen_encoder/dae/classification_report.csv']

In [55]:
records = []
for report in class_reports:
    path_parts = report.split('/')[3:-1]
    stage_1_pretrain_method = path_parts[0]
    stage_2_pretrain_method = path_parts[3]
    stage_2_pretrain_encoder_trained = path_parts[2] == 'no_freeze'
    
    finetune_encoder_trained = path_parts[1] == 'full_model'
    finetune_decoder_trained = path_parts[1] != 'linear_probe'
    
    
    cr_df = pd.read_csv(report, index_col='Unnamed: 0')
    accuracy = cr_df.loc['accuracy', 'support']
    precision_weighted = cr_df.loc['weighted avg', 'precision']
    precision_macro = cr_df.loc['macro avg', 'precision']
    recall_weighted = cr_df.loc['weighted avg', 'recall']
    recall_macro = cr_df.loc['macro avg', 'recall']
    f1_weighted = cr_df.loc['weighted avg', 'f1-score']
    f1_macro = cr_df.loc['macro avg', 'f1-score']
    
    tm_df = pd.read_csv(report.replace('classification_report.csv', 'test_metrics.csv'))
    loss = tm_df.loc[0, 'loss']
    weights_path = report.replace('classification_report.csv', 'best_model.pth').replace('./logs', './weights')
    assert os.path.exists(weights_path), f'Weights path {weights_path} does not exist'
    records.append({
        'stage_1_pretrain_method': stage_1_pretrain_method,
        'stage_2_pretrain_method': stage_2_pretrain_method,
        'stage_2_pretrain_encoder_trained': stage_2_pretrain_encoder_trained,
        'finetune_encoder_trained': finetune_encoder_trained,
        'finetune_decoder_trained': finetune_decoder_trained,
        'accuracy': accuracy,
        'precision_weighted': precision_weighted,
        'precision_macro': precision_macro,
        'recall_weighted': recall_weighted,
        'recall_macro': recall_macro,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro,
        'loss': loss,
        'weights_path': weights_path,
    })
df = pd.DataFrame(records)
df = df.sort_values(by=['stage_1_pretrain_method', 'stage_2_pretrain_method', 'stage_2_pretrain_encoder_trained', 'finetune_encoder_trained', 'finetune_decoder_trained'])
df = df.reset_index(drop=True)
df.to_csv('./logs/hires_simclr_tests_finetune2/finetune_results.csv', index=False)

In [44]:
df.to_csv('./logs/hires_simclr_tests_finetune2/finetune_results_old.csv', index=False)

In [ ]:
# test for significant differences in loss
df['stage_1_pretrain_method'] = df['stage_1_pretrain_method'].astype('category')
df['stage_2_pretrain_method'] = df['stage_2_pretrain_method'].astype('category')
df['stage_2_pretrain_encoder_trained'] = df['stage_2_pretrain_encoder_trained'].astype('category')
df['finetune_encoder_trained'] = df['finetune_encoder_trained'].astype('category')
df['finetune_decoder_trained'] = df['finetune_decoder_trained'].astype('category')
df['loss'] = df['loss'].astype(float)
df.sort_values(by=['loss']).iloc[1].weights_path

stage_1_pretrain_method                                                  hires_simclr
stage_2_pretrain_method                                                           dae
stage_2_pretrain_encoder_trained                                                False
finetune_encoder_trained                                                        False
finetune_decoder_trained                                                         True
accuracy                                                                     0.849135
precision_weighted                                                           0.860423
precision_macro                                                              0.730781
recall_weighted                                                              0.849135
recall_macro                                                                 0.773595
f1_weighted                                                                  0.852067
f1_macro                                              

In [5]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

model = ols('loss ~ stage_1_pretrain_method + stage_2_pretrain_method + stage_2_pretrain_encoder_trained + finetune_encoder_trained + finetune_decoder_trained', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table['sig. (a = 0.05)'] = (anova_table['PR(>F)'] < 0.05)
anova_table

,sum_sq,df,F,PR(>F),sig. (a = 0.05)
stage_1_pretrain_method,1.251519e+08,3.0,19.395112,1.038509e-06,True
stage_2_pretrain_method,7.859901e+06,2.0,1.827104,1.817168e-01,False
stage_2_pretrain_encoder_trained,1.082481e+07,1.0,5.032645,3.397705e-02,True
finetune_encoder_trained,1.107553e+06,1.0,0.514921,4.796684e-01,False
finetune_decoder_trained,7.339436e+08,1.0,341.223390,4.367519e-16,True
Residual,5.377296e+07,25.0,NaN,NaN,False
